# Block 2 — Introduction in Time Series Data: Preprocessing Fundamentals

**Goals for this block:**
- Implement effective strategies for handling missing values
- Normalize and standardize features for optimal model performance
- Create proper time-aware train/validation/test splits

## 0. Setup & Environment

Let's import the necessary libraries for time series analysis and visualization.

In [ ]:
# GUIDED: imports and environment checks
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Initial Exploration

We'll work with an **energy generation** dataset that contains typical patterns and challenges found in real-world time series data.

In [ ]:
# GUIDED: load data
df = pd.read_parquet("energy_synthetic.parquet")

In [ ]:
df.head(6)

In [ ]:
# parse a timestamp column smartly
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head(6)

In [ ]:
df.index.freq = 'h'  # hourly frequency

In [ ]:
df.set_index('timestamp', inplace=True)
df.head(6)

### 1.1 Data Quality Checks

Before preprocessing, we need to verify several important properties:
- **Index integrity**: Is the index properly formatted as a DatetimeIndex and monotonically increasing?
- **Frequency detection**: Can we detect the native sampling frequency of the data?
- **Data completeness**: Are there missing values or duplicate timestamps that need handling?

In [ ]:
# GUIDED: basic info
print(df.info())

# Check duplicates
dups = df.index.duplicated().sum()
print(f'Duplicate timestamps: {dups}')

# infer frequency
inferred = pd.infer_freq(df.index)
print('Inferred frequency:', inferred)

# Check missing
missing = df.isna().sum()
print('Missing values per column:\n', missing)


In [ ]:
# GUIDED: visualize raw series using matplotlib
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3))
df.iloc[:1000, 0].plot(title='first ~1000 points', xlabel='Timestamp (hour)', ylabel='Value (MW)')

## 2. Handling Missing Values in Time Series

Missing values in time series require special attention as they can affect temporal patterns. The appropriate strategy depends on your domain knowledge and the characteristics of the data:

- **Forward-fill**: Propagates the last valid observation forward (assumes persistence)
- **Backward-fill**: Uses next known values to fill gaps (useful for backfilling historical data)
- **Interpolation**: Creates smooth transitions between known values (linear, polynomial)

The choice should reflect the underlying physical or business process generating the data.

In [ ]:
print('Missing values per column:\n', missing)

In [ ]:
# GUIDED: choose a missing value strategy and apply it
# TODO: Replace 'method' / 'limit' as needed or implement interpolation

filled = df.copy()

# Example 1: linear-based interpolation
# filled = filled.interpolate(method='linear')

# Example 2: forward/backward fill
filled = filled.ffill().bfill()

In [ ]:
# EXERCISE: Compare filled vs non-filled data in subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot original resampled data (with missing values)
df.iloc[:1000, 0].plot(ax=ax1, title='Original resampled data (with missing values)', 
                               xlabel='Timestamp', ylabel='Value (MW)', color='blue')
ax1.grid(True, alpha=0.3)

# Plot filled data
filled.iloc[:1000, 0].plot(ax=ax2, title='After missing value handling', 
                           xlabel='Timestamp', ylabel='Value (MW)', color='orange')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# GUIDED: Check now many missing values remain
print('Missing values per column:\n', filled.isna().sum())

## 3. Resampling Time Series Data

**Goal:** Transform irregular or high-frequency data into a consistent, regular time grid.

1. Detect the original (native) sampling frequency
2. Choose an appropriate target frequency (hourly, daily, etc.)
   S: Seconds<br>
   min: Minutes<br>
   H: Hours<br>
   D: Day<br>
   W: Week<br>
   M: Month<br>
   Q: Quarter<br>
   Y: Year<br>

3. When downsampling (reducing the frequency), use the right aggregation method:
      - **Mean**: For measurements representing instantaneous values
      - **Sum**: For cumulative values that should be added (e.g., energy production)
      - **Last/First**: For snapshots or state-based values
   
4. When Upsampling (increasing the frequency), use the right filling method:
  - **Forward-fill**: Propagates the last valid observation forward (assumes persistence)
   - **Backward-fill**: Uses next known values to fill gaps (useful for backfilling historical data)
   - **Interpolation**: Creates smooth transitions between known values (linear, polynomial)


In [ ]:
# GUIDED: Downsampling

#resampled = filled.resample('D').mean()        # for instantaneous data
resampled = filled.resample('D').sum()         # for cumulative data
#resampled = filled.resample('D').last()        # for state-based data

# GUIDED: Upsampling
#resampled = filled.resample('15min').ffill().dropna()    # forward-fill
#resampled = filled.resample('15min').bfill().dropna()     # backward-fill
#resampled = filled.resample('15min').interpolate(method='linear').dropna()  #

resampled.head(3)

In [ ]:
# GUIDED: visualize resampled series
plt.figure(figsize=(10, 3))
resampled.iloc[:1000, 0].plot(title=f'Resampled series')
plt.tight_layout()
plt.show()

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Data Preparation</h2>

Complete the following tasks with the `de_energy_dataset.csv` dataset:
1. Load and visualize the raw data
2. Identify and handle missing values 
   - Try different filling methods (e.g., ffill, bfill, interpolate)

3. Resample the data
   - The data are monthly, try resampling to daily
   
   HINT: When upsampling always fill the missing values afterwards


This exercise will help reinforce the concepts of data cleaning and preparation for time series.
</div>

In [ ]:
# Load exercise data

df_excercise = ...
#df_excercise['timestamp'] = ...


In [ ]:
df_excercise.head(6)

In [ ]:
# visualize raw series: it should contain the different energy sources "Solar", "Wind", "Fossil"


In [ ]:
# Check missing values


In [ ]:
# handle missing values


In [ ]:
# Resampling: try frequency "D", "h", "W" and the appropriate aggregation / filling method


## 4. Train/Validation/Test Split (for time-series)

Unlike traditional random splits used in other ML tasks, time series data requires **chronological splitting** to preserve temporal dependencies and avoid look-ahead bias.

- **Hold-Out Split**: Simple chronological division (e.g., 70% train, 30% test)
- **Rolling Window**: Multiple train/test splits that move forward in time
- **Expanding Window**: Growing training set with a rolling evaluation window

For this introductory block, we'll implement a basic Hold-Out split to establish the foundation for more advanced techniques.

In [ ]:
# GUIDED: chronological split

len_train = int(len(resampled) * 0.7)

train = resampled.iloc[:len_train]
test = resampled.iloc[len_train:]

print('Split sizes:', len(train), len(test))
print('Ranges:')
print('Train:', train.index.min(), '→', train.index.max())
print('Test :', test.index.min(),  '→', test.index.max())

## 5. Rescaling

Proper scaling is essential for many machine learning algorithms to work effectively with time series data.

- **Z-score (StandardScaler)**: `(x - mean) / std` - Centers data around zero with unit variance
- **Min-Max (MinMaxScaler)**: Scales data to a fixed range [0, 1]
- **Robust Scaler**: Uses median and IQR, making it resistant to outliers

### ⚠️ Important: Avoid Data Leakage
Always fit scalers **only on training data** to prevent information leakage from the validation and test sets. The code below demonstrates the effect of scaling but should not be applied to the entire dataset in a real forecasting scenario.

In [ ]:
# GUIDED: Experiment with different scaling methods
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

scaler = StandardScaler()
# scaler = MinMaxScaler()
# scaler = RobustScaler()

train_scaled = pd.DataFrame(scaler.fit_transform(train), index=train.index, columns=train.columns)
test_scaled = pd.DataFrame(scaler.transform(test), index=test.index, columns=test.columns)


In [ ]:
# Visualize original vs scaled data in subplots

fig, ((ax1, ax2)) = plt.subplots(1, 2, figsize=(15, 5))
train.iloc[:1000, 0].plot(ax=ax1, title='Original Train Data (first ~1000 points)', 
                          xlabel='Timestamp', ylabel='Value (MW)', color='blue')
train_scaled.iloc[:1000, 0].plot(ax=ax2, title='Scaled Train Data (first ~1000 points)', 
                            xlabel='Timestamp', ylabel='Scaled Value', color='orange')

plt.show()

In [ ]:
# Saving the preprocessed data
train_scaled.to_parquet("train_preprocessed.parquet")
test_scaled.to_parquet("test_preprocessed.parquet")

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train-Validation-Test Split & Scaling</h2>

Apply what you've learned to the electricity dataset:
1. Split the resampled data into train, validation, and test sets using the Hold-Out Split method
2. Apply the most appropriate scaling method 
   - Try different scaling frequencies (e.g., zscore, MinMax)
   - Visualize random samples
3. Verify that scaling was properly applied without data leakage

This exercise will help you practice implementing a proper time series preprocessing pipeline.
</div>

In [ ]:
# split the resampled electricity data in train validation test using the Hold-Out Split method
# 70% train, 15% test




In [ ]:
# rescale the data using the most appropriate method

scaler = ...
train_scaled_ex = ...
test_scaled_ex = ...

In [ ]:
# visualize original vs scaled data in subplots


In [ ]:
# Saving the preprocessed data (save data with no resampling "resampled_exercise = filled_exercise")
train_scaled_ex.to_parquet("train_preprocessed_ex.parquet")
test_scaled_ex.to_parquet("test_preprocessed_ex.parquet")

## ✅ Summary

### What You've Accomplished:
- Loaded and performed quality checks on time series data
- Implemented strategies for handling missing values and resampling
- Created time-aware train/validation/test splits
- Applied proper scaling techniques while avoiding data leakage

These preprocessing steps are crucial for achieving good performance with any time series forecasting model.